# Gradient Descent — From Scratch in PyTorch

Implement gradient descent and its variants by hand on torch.Tensors, validate against torch.autograd, then compare to the idiomatic torch.optim API.

## Configuration

Device, seed, and dtype come from the repo's config.toml via shared.config.configure() — never hardcoded. On Apple Silicon this runs on mps.

In [1]:
import sys
from pathlib import Path

import torch
import matplotlib.pyplot as plt


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


HERE = Path.cwd()
REPO_ROOT = _find_repo_root(HERE)
sys.path.insert(0, str(HERE))
sys.path.insert(0, str(REPO_ROOT))

from gd.models import LinearModel
from gd.optimizers import SGD, Momentum, RMSProp, Adam
from shared.plotting import plot_loss_surface, plot_descent_path
from shared.config import configure

device = configure()  # reads config.toml -> device/seed/dtype
print(f"running on: {device}")


running on: mps


## The problem

A 1-feature linear regression y = w·x + b, so the loss surface over (w, b) is 2-D and plottable. Data lives on the configured device.

In [2]:
torch.manual_seed(0)
N = 64
X = torch.randn(N, 1, device=device)
true_w, true_b = 2.0, 0.5
y = X[:, 0] * true_w + true_b + 0.1 * torch.randn(N, device=device)
print("X", tuple(X.shape), "on", X.device.type)


X (64, 1) on mps


## The loss surface

The surface is a cheap visualization, so we scan it on CPU copies of the data; the actual training below runs on the configured device.

In [3]:
X_cpu, y_cpu = X.cpu(), y.cpu()


def loss_wb(w_val: float, b_val: float) -> float:
    m = LinearModel(w=torch.tensor([w_val]), b=torch.tensor(b_val))
    return m.loss(X_cpu, y_cpu).item()


ax = plot_loss_surface(loss_wb, w_range=(-1, 5), b_range=(-3, 4), steps=50)
ax.set_title("MSE loss surface")
plt.show()


/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_78614/706004290.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## From-scratch gradient descent

Manual MSE gradients, stepping the parameters ourselves on the configured device (immutable: a new model each step).

In [4]:
model = LinearModel(w=torch.zeros(1, device=device), b=torch.zeros((), device=device))
opt = SGD(lr=0.1)
path = [(model.w.item(), model.b.item())]
for _ in range(60):
    gw, gb = model.gradients(X, y)
    nw, nb = opt.step([model.w, model.b], [gw, gb])
    model = LinearModel(w=nw, b=nb)
    path.append((model.w.item(), model.b.item()))

ax = plot_loss_surface(loss_wb, w_range=(-1, 5), b_range=(-3, 4), steps=50)
plot_descent_path(path, ax=ax, color="white")
ax.set_title("SGD descent path")
plt.show()
print("final (w, b):", tuple(round(v, 4) for v in path[-1]), "| loss:", round(model.loss(X, y).item(), 5))


final (w, b): (2.021, 0.4773) | loss: 0.00973


/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_78614/1353212020.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Validate gradients against autograd

Our hand-derived gradients should match torch.autograd (both on the configured device).

In [5]:
w = torch.zeros(1, device=device, requires_grad=True)
b = torch.zeros((), device=device, requires_grad=True)
loss = LinearModel(w=w, b=b).loss(X, y)
loss.backward()
gw, gb = LinearModel(w=w.detach(), b=b.detach()).gradients(X, y)
assert torch.allclose(gw, w.grad, atol=1e-4)
assert torch.allclose(gb, b.grad, atol=1e-4)
print("manual gradients match autograd ✓")


manual gradients match autograd ✓


## Momentum vs RMSProp vs Adam

Four from-scratch optimizers on the same problem.

In [6]:
def run(opt, steps: int = 100) -> list[float]:
    model = LinearModel(w=torch.zeros(1, device=device), b=torch.zeros((), device=device))
    hist = [model.loss(X, y).item()]
    for _ in range(steps):
        gw, gb = model.gradients(X, y)
        nw, nb = opt.step([model.w, model.b], [gw, gb])
        model = LinearModel(w=nw, b=nb)
        hist.append(model.loss(X, y).item())
    return hist


for name, opt in {"SGD": SGD(0.1), "Momentum": Momentum(0.1), "RMSProp": RMSProp(0.1), "Adam": Adam(0.1)}.items():
    plt.plot(run(opt), label=name)
plt.xlabel("iteration")
plt.ylabel("MSE loss")
plt.yscale("log")
plt.legend()
plt.title("Optimizer convergence")
plt.show()


/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_78614/3463815472.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## The idiomatic PyTorch way

Reproduce the result with torch.optim.Adam — from-scratch and library agree.

In [7]:
model = LinearModel(w=torch.zeros(1, device=device), b=torch.zeros((), device=device))
opt = Adam(lr=0.05)
for _ in range(300):
    gw, gb = model.gradients(X, y)
    nw, nb = opt.step([model.w, model.b], [gw, gb])
    model = LinearModel(w=nw, b=nb)

w = torch.zeros(1, device=device, requires_grad=True)
b = torch.zeros((), device=device, requires_grad=True)
ref = torch.optim.Adam([w, b], lr=0.05)
for _ in range(300):
    ref.zero_grad()
    loss = torch.mean((X @ w + b - y) ** 2)
    loss.backward()
    ref.step()

assert torch.allclose(model.w, w.detach(), atol=1e-2)
assert torch.allclose(model.b, b.detach(), atol=1e-2)
print("from-scratch Adam matches torch.optim.Adam ✓")
print("scratch (w,b):", round(model.w.item(), 4), round(model.b.item(), 4))
print("torch   (w,b):", round(w.item(), 4), round(b.item(), 4))


from-scratch Adam matches torch.optim.Adam ✓
scratch (w,b): 2.021 0.4773
torch   (w,b): 2.021 0.4773


## Takeaways

- The gradient points uphill, so we step in the opposite direction to reduce the loss.
- The learning rate trades convergence speed against stability: too large and the optimizer overshoots; too small and training stalls.
- Momentum smooths the gradient signal by accumulating an exponential moving average of past updates, reducing oscillation.
- RMSProp adapts the learning rate per coordinate using the history of squared gradients, making it robust to different gradient scales.
- Adam combines both: momentum for direction and per-coordinate scaling for step size, with bias correction to handle cold-start.
- Device, seed, and dtype come from `config.toml` so the same code runs on mps/cuda/cpu unchanged.
